In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 加载 .env 文件中的环境变量（BASE_URL、API_KEY 等）
load_dotenv()
BASE_URL = os.getenv("BASE_URL")
API_KEY = os.getenv("API_KEY")


In [ ]:
# 初始化模型，使用 OpenAI 兼容接口接入 Qwen 模型
model = init_chat_model(
    model="Qwen/Qwen3-VL-30B-A3B-Instruct",
    model_provider="openai",
    base_url=BASE_URL,
    api_key=API_KEY,
)

In [ ]:
from langchain.tools import tool

# 使用 @tool 装饰器定义工具，docstring 会作为工具描述传给 LLM
@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b

In [ ]:
# 定义加法和除法工具
@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b

In [ ]:
# 汇总所有工具，并构建工具名称到工具实例的映射（供 tool_node 按名查找）
tools = [add, divide, multiply]
tools_by_name = {tool.name: tool for tool in tools}

# 将工具绑定到模型，LLM 便可在响应中发起工具调用
model_with_tools = model.bind_tools(tools)

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator


# 自定义 State，在消息列表基础上额外追踪 LLM 调用次数
# Annotated[list, operator.add] 表示多个节点返回的 messages 会自动追加合并，而非覆盖
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

In [ ]:
from langchain.messages import SystemMessage


def llm_call(state: dict):
    """LLM decides whether to call a tool or not"""

    # 将系统提示拼在历史消息前面，一起发给 LLM
    # LLM 会根据上下文决定：直接回答 or 发起工具调用
    return {
        "messages": [
            model_with_tools.invoke(
                [
                    SystemMessage(
                        content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
                    )
                ]
                + state["messages"]
            )
        ],
        # 每次进入此节点，调用计数 +1
        "llm_calls": state.get('llm_calls', 0) + 1
    }

In [ ]:
from langchain.messages import ToolMessage


def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    # 取出最后一条消息中 LLM 请求的所有工具调用
    for tool_call in state["messages"][-1].tool_calls:
        # 按名称找到对应工具并执行，得到观测结果
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        # 将执行结果包装为 ToolMessage，tool_call_id 用于与请求匹配
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END


def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]

    # 如果 LLM 的响应中包含工具调用请求，则路由到 tool_node 执行工具
    if last_message.tool_calls:
        return "tool_node"

    # 否则 LLM 已给出最终答案，结束图的执行
    return END

In [ ]:
# ── 构建图 ──────────────────────────────────────────────
agent_builder = StateGraph(MessagesState)

# 添加节点
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# 连接边：START → llm_call
agent_builder.add_edge(START, "llm_call")

# 条件边：llm_call → tool_node（有工具调用）或 END（无工具调用）
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]
)

# 工具执行完毕后，返回 llm_call 让 LLM 处理工具结果
agent_builder.add_edge("tool_node", "llm_call")

# 编译为可执行的图
agent = agent_builder.compile()

# ── 可视化图结构 ─────────────────────────────────────────
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

# ── 运行 Agent ──────────────────────────────────────────
from langchain.messages import HumanMessage
messages = [HumanMessage(content="Add 3 and 4.")]
messages = agent.invoke({"messages": messages})

# 打印所有消息（含中间的工具调用和结果）
for m in messages["messages"]:
    m.pretty_print()